In [ ]:
!pip install pyngrok python-socketio uvicorn nest_asyncio

In [ ]:
from pyngrok import ngrok
import socketio
import nest_asyncio
import uvicorn
import asyncio
from multiprocessing import Process
import httpx

LLAMA_API_URL = "http://localhost:1234/v1/chat/completions"

sio = socketio.AsyncServer(async_mode='asgi', cors_allowed_origins='*', max_http_buffer_size=8*1024*1024)
socket_app = socketio.ASGIApp(sio)

nest_asyncio.apply()
config = uvicorn.Config(app=socket_app, host="0.0.0.0", port=5555, log_level="info")
server = uvicorn.Server(config)

def run_server():
  loop = asyncio.get_event_loop()
  asyncio.set_event_loop(loop)
  loop.run_until_complete(server.serve())

@sio.on('use_model:vlm')
async def handle_vlm_task(sid, args):
    async with httpx.AsyncClient(timeout=None) as client:
    response = await client.post(LLAMA_API_URL, json=data)

    if response.status_code == 200:
        ai_res = response.json()['choices'][0]['message']['content']
        res = {"results": ai_res, 'task_id': args['task_id']}
        await sio.emit('vlm_res', res, to=sid)
    else:
        await sio.emit('error', f"Llama Error: {response.text}", to=sid)

In [ ]:
import subprocess

print("🧠 Starting Llama-Server...")
llama_cmd = [
    "./llama-server",
    "-m", VLM_PATH,
    "--mmproj", MMPROJ_PATH,
    "--host", "0.0.0.0",
    "--port", "1234",
    "-ngl", "99",
    "--no-mmap",
    "-c", "8192",
    "-b", "1024",
    "--ubatch-size", "4096",
    "-fa", "on",
    "-np", "1",
    "--cont-batching",
    "--jinja"
]

llama_proc = subprocess.Popen(
    llama_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=LLAMA_CWD
)

for line in llama_proc.stdout:
    print(f"[Llama]: {line.strip()}")
    if "HTTP server listening" in line or "main: server is listening" in line:
        print("\n✅ LLAMA IS READY!")
        break

ngrok.kill()
ngrok.set_auth_token("2v49o4k59WShfF0dsvpcDt7ooYW_3YTVRwPgcogMbvUjqXosn")
public_url = ngrok.connect(5555).public_url
print(f"🚀 NGROK LIVE AT: {public_url}")

server = uvicorn.Server(config)
print("🚀 Starting Socket.IO Server...")
asyncio.run(server.serve())